### Setup RAG with Milvus

In [2]:
# Import required libraries
import os
import json
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ibm import WatsonxEmbeddings 
from pymilvus import connections, utility, FieldSchema, CollectionSchema, DataType, Collection

In [3]:
# List of the URLs we will be extracting content from
urls = [
    "https://www.ibm.com/case-studies/us-open",
    "https://www.ibm.com/sports/usopen",
    "https://newsroom.ibm.com/US-Open-AI-Tennis-Fan-Engagement",
    "https://newsroom.ibm.com/2025-08-18-ibm-and-the-usta-roll-out-ai-powered-fan-experiences-for-2025-us-open",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

In [4]:
# Split the data in these documents to chunks
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
doc_splits = text_splitter.split_documents(docs_list)

In [5]:
# Initialize watsonx embeddings
credentials = {
    "url": os.getenv("WATSONX_URL"),
    "apikey": os.getenv("WATSONX_API_KEY")
}
project_id = os.getenv("WATSONX_PROJECT_ID")

embeddings_model = WatsonxEmbeddings(
    model_id="intfloat/multilingual-e5-large",
    url=credentials.get("url"),
    apikey=credentials.get("apikey"),
    project_id=project_id,
)

In [6]:
# Define milvus schema
COLLECTION_NAME = "rag_documents"
DIMENSION = 1024  
MILVUS_HOST = "localhost"
MILVUS_PORT = "19530"
print(f"Connecting to Milvus at {MILVUS_HOST}:{MILVUS_PORT}...")
connections.connect(host=MILVUS_HOST, port=MILVUS_PORT)
print("Connection successful.")

pk_field = FieldSchema(name="pk", dtype=DataType.INT64, is_primary=True, auto_id=True)
text_field = FieldSchema(name="chunk", dtype=DataType.VARCHAR, max_length=65535)
metadata_field = FieldSchema(name="metadata", dtype=DataType.VARCHAR, max_length=65535)
title_field = FieldSchema(name="title", dtype=DataType.VARCHAR, max_length=512)
source_field = FieldSchema(name="source_url", dtype=DataType.VARCHAR, max_length=512)
vector_field = FieldSchema(name="chunk_embeddings", dtype=DataType.FLOAT_VECTOR, dim=DIMENSION)
schema = CollectionSchema(
    fields=[pk_field, text_field, metadata_field, title_field, source_field, vector_field],
    description="RAG documents stored as vector embeddings"
)

Connecting to Milvus at localhost:19530...
Connection successful.


In [7]:
# Create the Collection
if utility.has_collection(COLLECTION_NAME):
    utility.drop_collection(COLLECTION_NAME)
    print(f"Old collection '{COLLECTION_NAME}' dropped.")

collection = Collection(name=COLLECTION_NAME, schema=schema)
print(f"New collection '{COLLECTION_NAME}' created.")


index_params = {
    "metric_type": "COSINE",  
    "index_type": "IVF_FLAT",
    "params": {"nlist": 1024}
}
collection.create_index(
    field_name="chunk_embeddings", 
    index_params=index_params
)
print("Index created successfully.")

Old collection 'rag_documents' dropped.
New collection 'rag_documents' created.
Index created successfully.


In [8]:
# Prepare and Insert the Data
texts = [doc.page_content for doc in doc_splits]
sources = [doc.metadata.get('source_url', 'Unknown') for doc in doc_splits]
metadata = [json.dumps(doc.metadata) for doc in doc_splits]
titles = [doc.metadata.get("title", "Unknown") for doc in doc_splits]

print(f"Generating embeddings for {len(texts)} chunks...")
vectors = embeddings_model.embed_documents(texts)
print("Embedding generation complete.")


data_to_insert = [
    texts,
    metadata,
    titles,
    sources,
    vectors
]
collection.insert(data_to_insert)
collection.flush()
collection.load()

print(f"\nSuccessfully inserted {collection.num_entities} vectors into '{COLLECTION_NAME}'.")

Generating embeddings for 54 chunks...
Embedding generation complete.

Successfully inserted 54 vectors into 'rag_documents'.


In [9]:
# Search the Milvus collection for a given query
def search_milvus(query: str, k: int = 3):
    try:
        collection = Collection(COLLECTION_NAME)
        collection.load()

        query_vector = embeddings_model.embed_query(query)
        search_vectors = [query_vector]
        search_params = {
            "metric_type": "COSINE",
            "params": {"nprobe": 10}
        }
        output_fields = ["chunk", "metadata", "title", "source_url"]

        results = collection.search(
            data=search_vectors, 
            anns_field="chunk_embeddings",
            param=search_params,
            limit=k,
            output_fields=output_fields
        )
        retrieved_documents = []
        for hit_list in results:
            for hit in hit_list:
                doc = {
                    "distance": hit.distance,
                    "title": hit.entity.get("title"),
                    "chunk": hit.entity.get("chunk"),
                    "source_url": hit.entity.get("source_url")
                }
                retrieved_documents.append(doc)    
        return retrieved_documents
        
    except Exception as e:
        print(f"An error occurred during Milvus search: {e}")
        return []

In [10]:
retrieved_chunks = search_milvus("Where was the 2025 US Open Tennis Championship held?")
for i, chunk in enumerate(retrieved_chunks):
    print(f"\n--- Chunk {i+1} (Distance: {chunk['distance']:.4f}) ---")
    print(f"Source: {chunk['source_url']}")
    print(f"Text Snippet: {chunk['chunk'][:50]}...")


--- Chunk 1 (Distance: 0.8136) ---
Source: Unknown
Text Snippet: The transformation of the US Open into a smarter b...

--- Chunk 2 (Distance: 0.8046) ---
Source: Unknown
Text Snippet: IBM and the USTA Roll Out AI-Powered Fan Experienc...

--- Chunk 3 (Distance: 0.7993) ---
Source: Unknown
Text Snippet: Here are some additional links:
The US Open is a s...
